In [ ]:
import pandas as pd

def load_data(sample, locus, reads_cutoff=2):
    fn = f"output/{sample}_Gr_bulk_RNA_{locus}/reads_{reads_cutoff}_u_1_l_0.01/alleles_by_umis.tsv"
    df = pd.read_csv(fn, sep="\t")
    df = df[df["mutation"].notna()].copy()
    df = (df
        .groupby('mutation', as_index=False)
        .agg(UMIs=('UMIs', 'sum'))
        .assign(sample_ID = sample)
        .sort_values('UMIs', ascending=False)
        )
    return df

def plot_umis_distribution(df, bins=20):
    import matplotlib.pyplot as plt
    import numpy as np
    # Log–log histogram (same idea as observed_count vs Count in the reference plot).
    x = df["UMIs"].astype(float)
    lo = max(float(x.min()), 1.0)
    hi = float(x.max())
    bins = np.logspace(np.log10(lo), np.log10(hi), bins)
    fig, ax = plt.subplots(figsize=(3, 2))
    ax.hist(
        x,
        bins=bins,
        edgecolor="white",
        linewidth=1,
        log=True,
    )
    ax.set_xscale("log")
    ax.set_xlabel("UMIs")
    ax.set_ylabel("Count")
    ax.set_title(sn)
    fig.tight_layout()
    plt.show()


## Preprocessing CA

In [ ]:
samples = ["LL583", "LL584", "LL638"]
locus = "CA"
df_list = []
for sn in samples:
    df = load_data(sample=sn, locus=locus)
    plot_umis_distribution(df)
    df_list.append(df)
df = pd.concat(df_list)
df.head()

In [ ]:
def merge_and_normalize_alleles(df):
    import numpy as np
    df_merge = (
        df
        .groupby('mutation', as_index=False)
        .agg(
            observed_count=('UMIs', 'sum'),
            sample_count=('sample_ID', 'count'),
            sample_id=('sample_ID', lambda x: ','.join(x.unique()))
        )
        .rename(columns={'mutation': 'allele'})
        .assign(
            allele_group=lambda x: np.where(x['allele'].eq('[]'), 'empty', 'non_empty')
        )
        .assign(
            normalized_count=lambda x: (
                x['observed_count'] /
                x.groupby('allele_group')['observed_count'].transform('sum')
            )
        )
        .sort_values('normalized_count', ascending=False)
        .drop(columns='allele_group')
    )
    return df_merge

df_merge = merge_and_normalize_alleles(df)
df_merge

In [ ]:
from matplotlib_venn import venn3
import matplotlib.pyplot as plt

df_set = [set(x['mutation']) for x in df_list]

plt.figure(figsize=(4, 3))
venn3(
    df_set, 
    samples, 
    alpha=0.4, 
    set_colors=('#66c2a5', '#fc8d62', '#8da0cb')
    )
plt.title(locus, fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
df_merge.to_csv("output/allele_bank_Gr_CA_ref.csv")

## Preprocessing RA

In [ ]:
samples = ["LL583", "LL584", "LL638"]
locus = "RA"
df_list = []
for sn in samples:
    df = load_data(sample=sn, locus=locus, reads_cutoff=1)
    plot_umis_distribution(df)
    df_list.append(df)
df = pd.concat(df_list)
df.head()

In [ ]:
df_merge = merge_and_normalize_alleles(df)
df_merge

In [ ]:
from matplotlib_venn import venn3
import matplotlib.pyplot as plt

df_set = [set(x['mutation']) for x in df_list]

plt.figure(figsize=(4, 3))
venn3(
    df_set, 
    samples, 
    alpha=0.4, 
    set_colors=('#66c2a5', '#fc8d62', '#8da0cb')
    )
plt.title(locus, fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
df_merge.to_csv("output/allele_bank_Gr_RA_ref.csv")

## Preprocessing TA

In [ ]:
samples = ["LL583", "LL584", "LL638"]
locus = "TA"
df_list = []
for sn in samples:
    df = load_data(sample=sn, locus=locus, reads_cutoff=2)
    plot_umis_distribution(df)
    df_list.append(df)
df = pd.concat(df_list)
df.head()

In [ ]:
df_merge = merge_and_normalize_alleles(df)
df_merge

In [ ]:
from matplotlib_venn import venn3
import matplotlib.pyplot as plt

df_set = [set(x['mutation']) for x in df_list]

plt.figure(figsize=(4, 3))
venn3(
    df_set, 
    samples, 
    alpha=0.4, 
    set_colors=('#66c2a5', '#fc8d62', '#8da0cb')
    )
plt.title(locus, fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
df_merge.to_csv("output/allele_bank_Gr_TA_ref.csv")

## Allele UMI distribution

In [ ]:
df_ref_CA = pd.read_csv("output/allele_bank_Gr_CA_ref.csv")
df_ref_TA = pd.read_csv("output/allele_bank_Gr_TA_ref.csv")
df_ref_RA = pd.read_csv("output/allele_bank_Gr_RA_ref.csv")

In [ ]:
(df_ref_CA['observed_count'].max(),
df_ref_TA['observed_count'].max(),
df_ref_RA['observed_count'].max())

In [ ]:
import numpy as np

x_var = np.logspace(np.log10(1), np.log10(5000), 20)
y_var_CC, x_var_CC = np.histogram(list(df_ref_CA["observed_count"]), bins=x_var)
y_var_TC, x_var_TC = np.histogram(list(df_ref_TA["observed_count"]), bins=x_var)
y_var_RC, x_var_RC = np.histogram(list(df_ref_RA["observed_count"]), bins=x_var)

fig, ax = plt.subplots(figsize=(4,3.5))
plt.loglog(x_var_CC[:-1], y_var_CC, "-o", label="CA", color='#1f77b4')
plt.loglog(x_var_TC[:-1], y_var_TC, "-o", label="TA", color='#ff7f0e')
plt.loglog(x_var_RC[:-1], y_var_RC, "-o", label="RA",color='#279e68')
plt.legend()
#plt.xlim([0,1100])
plt.xlabel('Allele UMI count '+r'$X$')
plt.ylabel("Frequency")
plt.tight_layout()

## Calculate Homoplasy Prob: $P$

### Curve fitting (CA / TA / RA)

**Data.** Alleles are assigned to UMI-count bins (`map_range`). For each locus, bins are aggregated so each row has a representative **allele UMI count** `observed_count` ($X$) and the bin-wise mean of **`raw_homoplasy`** in $[0,1]$. Fitting uses these paired observations $(X_i, y_i)$.

**Model.** We assume a monotone saturation curve with asymptotes near 0 and 1. Because the diagnostic plot uses a **logarithmic** $X$-axis, the linear predictor is taken on $\log_{10} X$ (with $X$ clipped to at least 1 so $\log_{10}$ is finite):

$$
P(X)=\frac{1}{1+\exp\bigl(-k\,(\log_{10}X-\theta)\bigr)}.
$$

Here $\theta=\log_{10}X_{50}$ is the $\log_{10}$ UMI count where $P\approx\frac{1}{2}$, and $k>0$ sets how steeply $P$ rises. This is a **two-parameter logistic** as a function of $\log_{10}X$.

**Estimation.** For each of CA, TA, and RA, $(\theta,k)$ is estimated by **nonlinear least squares** (`scipy.optimize.curve_fit`), minimizing $\sum_i\bigl(y_i-P(X_i)\bigr)^2$. The starting $\theta$ is the median of $\log_{10}X_i$ over bins; **bounded optimization** keeps $\theta$ and $k$ in a plausible range.

**Plotting.** Overlay curves are sampled on a **dense** `np.logspace` grid so lines look smooth on log-scaled axes (connecting only bin centers would look kinked).

**Downstream.** `infer_prob_CC`, `infer_prob_TC`, and `infer_prob_RC` evaluate $P(X)$ for arbitrary counts using the fitted $(\theta,k)$ per locus.

**Caveats.** Bin means weight alleles implicitly through the binning scheme; there are **no explicit observation-count weights** in the fit. If you later have successes/totals per bin, a **binomial (logistic) GLM** with weights would be statistically cleaner than unweighted least squares on bin means.

In [ ]:
def map_range(x,unit=1.3):
    a=np.log(x)/np.log(unit)
    a_floor=np.floor(a)
    a_upper=np.ceil(a)
    if a<=9:
        return x
    else:
        return unit**a_upper

In [ ]:
df_ref_CA = pd.read_csv("output/allele_bank_Gr_CA_ref.csv", index_col=0)
df_ref_CA['bin'] = df_ref_CA['observed_count'].apply(map_range)
df_ref_CA['raw_homoplasy'] = df_ref_CA['sample_count'].apply(lambda x: 0 if x==1 else 1)
# df_ref_CA.head()

df_ref_RA = pd.read_csv("output/allele_bank_Gr_RA_ref.csv", index_col=0)
df_ref_RA['bin'] = df_ref_RA['observed_count'].apply(map_range)
df_ref_RA['raw_homoplasy'] = df_ref_RA['sample_count'].apply(lambda x: 0 if x==1 else 1)

df_ref_TA = pd.read_csv("output/allele_bank_Gr_TA_ref.csv", index_col=0)
df_ref_TA['bin'] = df_ref_TA['observed_count'].apply(map_range)
df_ref_TA['raw_homoplasy'] = df_ref_TA['sample_count'].apply(lambda x: 0 if x==1 else 1)


In [ ]:
df_sample_CA = (
    df_ref_CA
    .groupby('bin')
    .agg({
        'observed_count': 'mean',
        'sample_count': 'mean',
        'raw_homoplasy': 'mean'
    })
)

df_sample_RA = (
    df_ref_RA
    .groupby('bin')
    .agg({
        'observed_count': 'mean',
        'sample_count': 'mean',
        'raw_homoplasy': 'mean'
    })
)

df_sample_TA = (
    df_ref_TA
    .groupby('bin')
    .agg({
        'observed_count': 'mean',
        'sample_count': 'mean',
        'raw_homoplasy': 'mean'
    })
)

In [ ]:
key = "raw_homoplasy"

import numpy as np
from scipy.optimize import curve_fit


def logistic_log10(x, log_x50, k):
    """Allele count X → [0, 1]; linear predictor uses log10(X) (matches log-scaled x-axis)."""
    x = np.maximum(np.asarray(x, dtype=float), 1.0)
    t = np.log10(x) - log_x50
    return 1.0 / (1.0 + np.exp(-k * t))


def fit_homoplasy_logistic(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y) & (x > 0)
    x, y = x[m], y[m]
    log_x50_0 = float(np.median(np.log10(np.maximum(x, 1.0))))
    popt, _ = curve_fit(
        logistic_log10,
        x,
        y,
        p0=[log_x50_0, 4.0],
        bounds=([-4.0, 0.25], [5.0, 80.0]),
        maxfev=50000,
    )
    return popt


xdata_CC = df_sample_CA["observed_count"].to_numpy()
ydata_CC = np.asarray(df_sample_CA[key])
popt_CC = fit_homoplasy_logistic(xdata_CC, ydata_CC)

xdata_TC = df_sample_TA["observed_count"].to_numpy()
ydata_TC = np.asarray(df_sample_TA[key])
popt_TC = fit_homoplasy_logistic(xdata_TC, ydata_TC)

xdata_RC = df_sample_RA["observed_count"].to_numpy()
ydata_RC = np.asarray(df_sample_RA[key])
popt_RC = fit_homoplasy_logistic(xdata_RC, ydata_RC)


In [ ]:
def infer_prob_CC(x):
    return logistic_log10(x, *popt_CC)


def infer_prob_TC(x):
    return logistic_log10(x, *popt_TC)


def infer_prob_RC(x):
    return logistic_log10(x, *popt_RC)

In [ ]:
import numpy as np
import seaborn as sns

key = "raw_homoplasy"

fig, ax = plt.subplots(figsize=(4, 3.5))
sns.scatterplot(x=df_sample_CA["observed_count"], y=df_sample_CA[key], label="CA", ax=ax)
sns.scatterplot(x=df_sample_TA["observed_count"], y=df_sample_TA[key], label="TA", ax=ax)
sns.scatterplot(x=df_sample_RA["observed_count"], y=df_sample_RA[key], label="RA", ax=ax)

x_plot = np.logspace(0, np.log10(1000), 400)
ax.plot(x_plot, logistic_log10(x_plot, *popt_CC), "-", color="#1f77b4", lw=1.8)
ax.plot(x_plot, logistic_log10(x_plot, *popt_TC), "-", color="#ff7f0e", lw=1.8)
ax.plot(x_plot, logistic_log10(x_plot, *popt_RC), "-", color="#279e68", lw=1.8)

ax.set_xscale("log")
ax.set_xlim(1, 1000)
ax.set_xlabel("Allele UMI count " + r"$X$")
ax.set_ylabel('Obs. homoplasy prob. '+r'$P$')
fig.tight_layout()

In [ ]:
df_ref_CA['smoothed_homoplasy'] = df_ref_CA['observed_count'].apply(infer_prob_CC)
df_ref_CA.loc[df_ref_CA['smoothed_homoplasy'] < 0, 'smoothed_homoplasy'] = 0

df_ref_TA['smoothed_homoplasy'] = df_ref_TA['observed_count'].apply(infer_prob_TC)
df_ref_TA.loc[df_ref_TA['smoothed_homoplasy'] < 0, 'smoothed_homoplasy'] = 0

df_ref_RA['smoothed_homoplasy'] = df_ref_RA['observed_count'].apply(infer_prob_RC)
df_ref_RA.loc[df_ref_RA['smoothed_homoplasy'] < 0, 'smoothed_homoplasy'] = 0

In [ ]:
df_ref_CA.head()

In [ ]:
fig,ax=plt.subplots(figsize=(4,3.5))
colors=list(sns.color_palette().as_hex())
key_2='smoothed_homoplasy'
key_1='normalized_count'
plt.plot(df_ref_CA[key_1],df_ref_CA[key_2],'-',lw=1.8,label='CA',color="#1f77b4")
plt.plot(df_ref_TA[key_1],df_ref_TA[key_2],'-',lw=1.8,label='TA',color="#ff7f0e")
plt.plot(df_ref_RA[key_1],df_ref_RA[key_2],'-',lw=1.8,label='RA',color="#279e68")
plt.legend()
plt.ylabel('Obs. homoplasy prob. '+r'$P$')
plt.xlabel('Cumulative UMI fraction '+r'$\zeta$')
plt.tight_layout()
plt.xscale('log')
# plt.yscale('log')

In [ ]:
alleles = [
    df_ref_CA['allele'].nunique()/1e3,
    df_ref_TA['allele'].nunique()/1e3,
    df_ref_RA['allele'].nunique()/1e3,
]
fig, ax = plt.subplots(figsize=(2.5, 2))
colors = ['#1f77b4', '#ff7f0e', '#279e68']
plt.bar(['CA','TA','RA'], alleles, color=colors)
plt.ylabel('Number of alleles (k)')
plt.tight_layout()


In [ ]:
observed_counts = [
    df_ref_CA['observed_count'].sum()/1e3,
    df_ref_TA['observed_count'].sum()/1e3,
    df_ref_RA['observed_count'].sum()/1e3,
]
fig, ax = plt.subplots(figsize=(2.5, 2))
colors = ['#1f77b4', '#ff7f0e', '#279e68']
plt.bar(['CA','TA','RA'], observed_counts, color=colors)
plt.ylabel('Number of UMIs (k)')
plt.tight_layout()


In [ ]:
df_ref_TA.head()

In [ ]:
cols = [
    'allele',
    'observed_count',
    'sample_count',
    'sample_id',
    'normalized_count',
    'smoothed_homoplasy'
]
df_ref_CA_save = df_ref_CA[cols]
df_ref_CA_save.to_csv("output/allele_bank_Gr_CA.csv.gz", index=False)

df_ref_RA_save = df_ref_RA[cols]
df_ref_RA_save.to_csv("output/allele_bank_Gr_RA.csv.gz", index=False)

df_ref_TA_save = df_ref_TA[cols]
df_ref_TA_save.to_csv("output/allele_bank_Gr_TA.csv.gz", index=False)